In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import cobra
import pandas

# Setting flux constraints based on biological context

The constraints of exchange reactions were set to resemble the platelet model iAT-PLT-636 (https://doi.org/10.1038/srep03925), i.e., metabolite uptake/excretion in MitoCore was blocked if no exchange reaction for that metabolite was present in the platelet model. Furthermore, the maximal uptake or excretion constraints for Glucose, Acetate and Lactate were set to the upper bounds determined from 13C analysis of resting platelets (https://doi.org/10.1016/j.ymben.2021.12.007).
For both models a source for glucose-6-phosphate was introduced to model utilization of glycogen stores. The upper limit for this “glycogen supply” was set to the upper limit determined from the same 13C analysis (https://doi.org/10.1016/j.ymben.2021.12.007).

In [3]:
unparameterized_models_path_dict = {
    'Mitocore_Original': './../unparameterized_models/Mitocore_Original.xml',
    'Mitocore_Preliminary': './../unparameterized_models/Mitocore_Preliminary.xml',
    'Mitocore_MitoMammal': './../unparameterized_models/Mitocore_MitoMammal.xml',
    'Mitocore_aligned_to_Human1': './../unparameterized_models/Mitocore_aligned_to_Human1.xml',
}

unparameterized_models_dict = {}

for key, path in unparameterized_models_path_dict.items():
    unparameterized_models_dict[key] = cobra.io.read_sbml_model(path)

# Insertion of a glycogen reaction
For both models a source for glucose-6-phosphate was introduced to model utilization of glycogen stores. The upper limit for this “glycogen supply” was set to the upper limit determined from the same 13C analysis (https://doi.org/10.1016/j.ymben.2021.12.007).

In [4]:
for model in unparameterized_models_dict.values():
    # add external and internal glycogen
    glyc_c = cobra.Metabolite(
        'glycogen_c',
        name='glycogen',
        compartment='c')
    glyc_e = cobra.Metabolite(
        'glycogen_e',
        name='glycogen',
        compartment='e')

    # add transport reaction for glycogen
    reaction = cobra.Reaction('Glycogen_t', name='Glycogen transport', lower_bound=-1000, upper_bound=1000)
    reaction.add_metabolites({glyc_e: -1, glyc_c: 1}) # external glycogen --> internal glycogen
        
    model.add_reactions([reaction])
    model.add_boundary(model.metabolites.get_by_id("glycogen_e"), type="exchange", lb=-2.35 * 0.06, ub=0.0) # 2.35 µmol/gdw min * 0.06 mmol/µmol / h/min = 2.35 * 0.06 mmol/gdw h

In [5]:
unparameterized_models_dict['Mitocore_aligned_to_Human1'].metabolites.get_by_id("glycogen_e")

Metabolite identifier,glycogen_e
Name,glycogen
Memory address,0x0239acf2e740
Formula,None
Compartment,e
In 2 reaction(s),"Glycogen_t, EX_glycogen_e"


Import parameters from xlsx

In [6]:
default_params_and_fba = pandas.read_excel(
    "./../model_preparation/plt_model_constraints.xlsx", index_col=0
)

In [7]:
def set_constraints(model, condition, write_path=''):

    print(condition)

    for reaction in model.reactions:
        reaction_id = "R_" + reaction.id

        try:
            default_params_and_fba.loc[reaction_id, condition + "_lb in µmol/min/gDW"]
        except:
            print(f"Couldn't find {reaction_id} in data.")
            continue
        
        lb = float(default_params_and_fba.loc[reaction_id, condition + "_lb in µmol/min/gDW"])
        ub = float(default_params_and_fba.loc[reaction_id, condition + "_ub in µmol/min/gDW"])

        if not lb == -1000: # keep default constraints
            lb = lb * 0.06 # update flux unit to mmol/gdw/h
            
        if not ub == 1000: 
            ub = ub * 0.06

        # check if any ubs are -1000 or lbs are 1000 (just to be save)
        if reaction.upper_bound == -1000 or reaction.lower_bound == 1000:
            print(reaction.id)

        reaction.bounds = (lb, ub)

    solution = model.optimize()
    
    print('ATP production')
    print(solution.fluxes['OF_ATP_MitoCore'])
    
    if not (write_path == ''):
        cobra.io.write_sbml_model(model, write_path)

CI GPR: ENSG000001152863 and ENSG00000110717 and ENSG00000178127 and ENSG00000213619 and ENSG00000158864 and ENSG00000167792 and ENSG00000023228 and ENSG00000198888 and ENSG00000198763 and ENSG00000198840 and ENSG00000198886 and ENSG00000212907 and ENSG00000198786 and ENSG00000198695 and ENSG00000145494 and ENSG00000184752 and ENSG00000164258 and ENSG00000139180 and ENSG00000004779 and ENSG00000131495 and ENSG00000119013 and ENSG00000128609 and ENSG00000184983 and ENSG00000174886 and ENSG00000147123 and ENSG00000168653 and ENSG00000065518 and ENSG00000186010 and ENSG00000099795 and ENSG00000119421 and ENSG00000147684 and ENSG00000140990 and ENSG00000166136 and ENSG00000151366 and ENSG00000090266 and ENSG00000267855 and ENSG00000170906 and ENSG00000136521 and ENSG00000183648 and ENSG00000109390 and ENSG00000130414 and (ENSG00000189043 or ENSG00000185633) and ENSG00000160194 and ENSG00000165264

CIV GPR: ENSG00000198804 and ENSG00000198712 and ENSG00000198938 and (ENSG00000131143 or ENSG00000131055) and ENSG00000178741 and ENSG00000135940 and (ENSG00000111775 or ENSG00000156885) and (ENSG00000126267 or ENSG00000160471) and ENSG00000164919 and (ENSG00000161281 or ENSG00000112695 or ENSG00000115944) and ENSG00000131174 and ENSG00000127184 and (ENSG00000176340 or ENSG00000187581)

In [8]:
CI_MitoCore_static_and_part = ['ENSG000001152863',
 'ENSG00000110717',
 'ENSG00000178127',
 'ENSG00000213619',
 'ENSG00000158864',
 'ENSG00000167792',
 'ENSG00000023228',
 'ENSG00000198888',
 'ENSG00000198763',
 'ENSG00000198840',
 'ENSG00000198886',
 'ENSG00000212907',
 'ENSG00000198786',
 'ENSG00000198695',
 'ENSG00000145494',
 'ENSG00000184752',
 'ENSG00000164258',
 'ENSG00000139180',
 'ENSG00000004779',
 'ENSG00000131495',
 'ENSG00000119013',
 'ENSG00000128609',
 'ENSG00000184983',
 'ENSG00000174886',
 'ENSG00000147123',
 'ENSG00000168653',
 'ENSG00000065518',
 'ENSG00000186010',
 'ENSG00000099795',
 'ENSG00000119421',
 'ENSG00000147684',
 'ENSG00000140990',
 'ENSG00000166136',
 'ENSG00000151366',
 'ENSG00000090266',
 'ENSG00000267855',
 'ENSG00000170906',
 'ENSG00000136521',
 'ENSG00000183648',
 'ENSG00000109390',
 'ENSG00000130414',
 'ENSG00000160194',
 'ENSG00000165264']
CI_Mitocore_or_parts = ['ENSG00000189043', 'ENSG00000185633']

CIV_MitoCore_static_and_part = ['ENSG00000198804',
 'ENSG00000198712',
 'ENSG00000198938',
 'ENSG00000178741',
 'ENSG00000135940',
 'ENSG00000164919',
 'ENSG00000131174',
 'ENSG00000127184']
CIV_MitoCore_or_parts = [["ENSG00000131143", "ENSG00000131055"], ["ENSG00000111775", "ENSG00000156885"], ["ENSG00000126267", "ENSG00000160471"], ["ENSG00000161281", "ENSG00000112695", "ENSG00000115944"], ["ENSG00000176340", "ENSG00000187581"]]


In [9]:
import itertools

or_combinations = list(itertools.product(*CIV_MitoCore_or_parts))

CIV_Mitocore_gpr = "("

for combination in or_combinations:
    CIV_Mitocore_gpr += " and ".join(CIV_MitoCore_static_and_part + list(combination)) + ") or ("

CIV_Mitocore_gpr = CIV_Mitocore_gpr[:-5] # remove last or

CI_Mitocore_gpr = "("
for or_part in CI_Mitocore_or_parts:
    CI_Mitocore_gpr += " and ".join(CI_MitoCore_static_and_part + [or_part]) + ") or ("
CI_Mitocore_gpr = CI_Mitocore_gpr[:-5] # remove last or

In [10]:
# set constraints and compare default and neutrophil model
for model_name, model in unparameterized_models_dict.items():
    with model:
        if model_name == 'Mitocore_MitoMammal' or model_name == 'Mitocore_aligned_to_Human1':
            model.reactions.get_by_id("CI_MitoCore").gene_reaction_rule = CI_Mitocore_gpr
            model.reactions.get_by_id("CIV_MitoCore").gene_reaction_rule = CIV_Mitocore_gpr
        set_constraints(model, 'plt', write_path="./../parameterized_plt_models/" + model_name + "_plt.xml")

plt
Couldn't find R_Glycogen_t in data.
Couldn't find R_EX_glycogen_e in data.
ATP production
6.521542498684816
plt
Couldn't find R_Glycogen_t in data.
Couldn't find R_EX_glycogen_e in data.
ATP production
6.521542498684816


c:\Users\Emanu\anaconda3\envs\smoment\lib\site-packages\cobra\core\reaction.py:489: UserWarning: Context management not implemented for gene reaction rules.
  warn("Context management not implemented for gene reaction rules.")


plt
Couldn't find R_CBPS in data.
Couldn't find R_ASPCT in data.
Couldn't find R_DHORTS in data.
Couldn't find R_DHORD9 in data.
Couldn't find R_DM_orot_c in data.
Couldn't find R_Glycogen_t in data.
Couldn't find R_EX_glycogen_e in data.
ATP production
6.521542498684816
plt
Couldn't find R_CBPS in data.
Couldn't find R_ASPCT in data.
Couldn't find R_DHORTS in data.
Couldn't find R_DHORD9 in data.
Couldn't find R_DM_orot_c in data.
Couldn't find R_Glycogen_t in data.
Couldn't find R_EX_glycogen_e in data.
ATP production
6.521542498684816
